In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.window import *
from dataclasses import dataclass
from pyspark.sql import Column, DataFrame, SparkSession
from collections.abc import Callable

In [0]:
@dataclass
class BronzeLayer:
    # Read configuration
    file_format:str
    schema_evolution_mode:str
    header:bool
    infer_column_types:bool
    schema_location:str
    load_location:str
    path_file:str

    # Write configuration
    output_mode:str
    checkpoint_location:str
    trigger:dict[str,str]
    merge_schema:str
    table_bronze:str
    def read_stream(self,spark) -> DataFrame:
       df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", self.file_format)
            .option("header", str(self.header).lower())
            .option("cloudFiles.inferColumnTypes", str(self.infer_column_types).lower())
            .option("cloudFiles.schemaEvolutionMode", self.schema_evolution_mode)
            .option("cloudFiles.schemaLocation", f"{self.schema_location}")
            .option("pathGlobFilter", self.path_file)
            .option("mode", "PERMISSIVE")
            .option("rescuedDataColumn", "_rescued_data")
            .option("cloudFiles.schemaHints", "_corrupt_record STRING")
            .option("columnNameOfCorruptRecord", "_corrupt_record")
            .load(f"{self.load_location}") 
            )
       return (
            df
            .withColumn("_file_name", col("_metadata.file_name"))
            .withColumn("_file_path", col("_metadata.file_path"))
            .withColumn("_file_size", col("_metadata.file_size"))
            .withColumn("_file_mod", col("_metadata.file_modification_time"))
            .withColumn("_processing_time", current_timestamp())
        )
    def write_stream(self,df):
        return (
            df.writeStream
            .format("delta")
            .outputMode(self.output_mode)
            .option("mergeSchema",str(self.merge_schema).lower())
            .option("checkpointLocation", self.checkpoint_location)
            .trigger(**self.trigger)
            .toTable(self.table_bronze)
        )

In [0]:
def get_reason(df: DataFrame) -> DataFrame:
    control_col = [
        name for name in df.columns if name.startswith("_is_")
    ]
    data_col = [
        name for name in df.columns
        if name not in control_col and name != "_sk"
    ]
    or_statement = " OR ".join([
        name for name in control_col
    ])

    return (
        df
        .filter(or_statement)
        .melt(
            ids=[*data_col, "_sk"],
            values=control_col,
            variableColumnName="reason",
            valueColumnName="status",
        )
        .filter(col("status") == True)
        .groupBy(*data_col, "_sk")
        .agg(collect_list("reason").alias("reason"))
    )

In [0]:
@dataclass
class SilverLayer:
    bronze_table_name: str
    silver_table_name: str
    bad_record_table_name: str
    schema_detail: dict[str, str]
    keys: list[str]

    def __post_init__(self) -> None:
        self.data_col = list(self.schema_detail.keys())
        self.control_col = [
            "source_file", "_processing_time",
            "csv_corrupt_record", "rescued_data",
        ]
        self.invalid_rule = {
            "int": r"^[+-]?[0-9]+$",
            "bigint": r"^[+-]?[0-9]+$",
            "decimal": r"^[+-]?([0-9]+([.][0-9]*)?|[.][0-9]+)([eE][+-]?[0-9]+)?$",
            "date": r"^([0-9]{4}-[0-9]{2}-[0-9]{2}|[0-9]{1,2}/[0-9]{1,2}/[0-9]{4})$",
        }

    def read_add_sk_from_bronze_table(self, batch_df: DataFrame) -> DataFrame:
        # Keep values as text until validation is complete.
        select_statement = [
            trim(col(name).cast("string")).alias(name)
            for name in self.data_col
        ]
        bronze_df = (
            batch_df
            .select(
                *select_statement,
                col("_file_path").alias("source_file"),
                "_processing_time",
                col("_corrupt_record").alias("csv_corrupt_record"),
                col("_rescued_data").alias("rescued_data"),
            )
        )
        # Normalize whole numbers inferred as decimals, such as 12.0.
        for name, data_type in self.schema_detail.items():
            if data_type.lower() in ["int", "bigint"]:
                bronze_df = bronze_df.withColumn(
                    name, regexp_replace(col(name), "[.]0+$", "")
                )

        # Stable row numbering lets independently evaluated checks join by _sk.
        partition_by_all = Window.orderBy(*self.data_col, *self.control_col)
        return (
            bronze_df
            .withColumn("_sk", row_number().over(partition_by_all))
        )

    def get_key_null_record(self, bronze_df: DataFrame) -> DataFrame:
        key_null_statement = {
            f"_is_{name}_null": col(name).isNull() | (col(name) == "")
            for name in self.keys
        }
        return (
            bronze_df
            .withColumns(key_null_statement)
            .transform(get_reason)
        )

    def get_invalid_record(self, bronze_df: DataFrame) -> DataFrame:
        invalid_col = {}

        for name, data_type in self.schema_detail.items():
            missing = col(name).isNull() | (col(name) == "")
            cast_value = col(name).try_cast(data_type)
            rule_type = data_type.lower().split("(")[0]

            if rule_type == "date":
                cast_value = coalesce(
                    cast_value,
                    try_to_timestamp(col(name), lit("M/d/yyyy")).cast("date"),
                )

            invalid = cast_value.isNull()
            if rule_type in self.invalid_rule:
                invalid = invalid | ~col(name).rlike(self.invalid_rule[rule_type])

            invalid_col[f"_is_{name}_invalid"] = coalesce(
                ~missing & invalid, lit(False)
            )
            if name not in self.keys:
                invalid_col[f"_is_{name}_null"] = missing

        business_col = {
            "_is_order_id_not_positive": col("order_id").try_cast("bigint") <= 0,
            "_is_quantity_not_positive": col("quantity").try_cast("int") <= 0,
            "_is_unit_price_negative": col("unit_price").try_cast("decimal(18,2)") < 0,
            "_is_total_amount_negative": col("total_amount").try_cast("decimal(18,2)") < 0,
            "_is_amount_mismatch": (
                col("total_amount").try_cast("decimal(18,2)")
                != col("unit_price").try_cast("decimal(18,2)")
                * col("quantity").try_cast("int")
            ),
            "_is_processing_time_invalid": col("_processing_time").isNull(),
            "_is_csv_invalid": col("csv_corrupt_record").isNotNull(),
            "_is_schema_invalid": col("rescued_data").isNotNull(),
        }

        return (
            bronze_df
            .withColumns({**invalid_col, **business_col})
            .transform(get_reason)
        )

    def get_dup_record(
        self, bronze_df: DataFrame, key_null_df: DataFrame
    ) -> DataFrame:
        partition_by_all = (
            Window.partitionBy(*self.data_col, "_processing_time")
            .orderBy("_sk")
        )
        partition_by_key = Window.partitionBy(*self.keys, "_processing_time")

        bronze_not_null_df = (
            bronze_df
            .join(key_null_df.select("_sk"), ["_sk"], "left_anti")
        )

        is_row_duplicate_df = (
            bronze_not_null_df
            .withColumn("rn", row_number().over(partition_by_all))
            .filter(col("rn") > 1)
            .drop("rn")
            .withColumn("reason", array(lit("_row_duplicate")))
        )

        is_key_duplicate_df = (
            bronze_not_null_df
            .join(is_row_duplicate_df.select("_sk"), ["_sk"], "left_anti")
            .withColumn("count", count("*").over(partition_by_key))
            .filter(col("count") > 1)
            .drop("count")
            .withColumn("reason", array(lit("_key_duplicate")))
        )

        return (
            is_row_duplicate_df
            .unionByName(is_key_duplicate_df)
        )

    def get_all_bad_record(
        self, invalid_df: DataFrame, key_null_df: DataFrame, duplicate_df: DataFrame
    ) -> DataFrame:
        return (
            invalid_df
            .unionByName(key_null_df)
            .unionByName(duplicate_df)
            .groupBy(*self.data_col, *self.control_col, "_sk")
            .agg(
                sort_array(array_distinct(flatten(collect_list("reason"))))
                .alias("reason")
            )
        )

    def get_final_result(
        self, bronze_df: DataFrame, all_bad_df: DataFrame
    ) -> DataFrame:
        cast_statement = []
        for name, data_type in self.schema_detail.items():
            cast_value = col(name).try_cast(data_type)
            if data_type.lower() == "date":
                cast_value = coalesce(
                    cast_value,
                    try_to_timestamp(col(name), lit("M/d/yyyy")).cast("date"),
                )
            cast_statement.append(cast_value.alias(name))

        partition_by_key = Window.partitionBy(*self.keys)
        return (
            bronze_df
            .withColumn("latest_processing_time", max("_processing_time").over(partition_by_key))
            .filter(col("_processing_time") == col("latest_processing_time"))
            .join(all_bad_df.select("_sk"), ["_sk"], "left_anti")
            .select(*cast_statement, "source_file", "_processing_time")
        )

    def load_bad_record(self, all_bad_df: DataFrame) -> None:
        bad_df = all_bad_df.drop("_sk")
        merge_col = [*self.data_col, *self.control_col, "reason"]
        merge_condition = " AND ".join([
            f"source.`{name}` <=> target.`{name}`" for name in merge_col
        ])
        (
            DeltaTable.forName(bad_df.sparkSession, self.bad_record_table_name)
            .alias("target")
            .merge(bad_df.distinct().alias("source"), merge_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )

    def load_to_silver_layer(self, final_result_df: DataFrame) -> None:
        merge_condition = " AND ".join([
            f"source.`{key}` = target.`{key}`" for key in self.keys
        ])
        (
            DeltaTable.forName(final_result_df.sparkSession, self.silver_table_name)
            .alias("target")
            .merge(final_result_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll(
                condition=(
                    col("source._processing_time") > col("target._processing_time")
                )
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
